# ReAct Pattern: The Reasoning Engine

The **ReAct (Reason + Act)** pattern is the foundational logic that separates simple chatbots from autonomous agents. It enables a Large Language Model (LLM) to interleave reasoning traces with task-specific actions—allowing the model to "think" about what to do, execute an action, observe the result, and refine its next step based on that feedback.

## 1. The Evolution of Autonomy
Reasoning in LLMs has evolved through three distinct stages:

1. **Direct Completion**: The model provides an answer immediately. (High risk of hallucination for calculation/logic).
2. **Chain of Thought (CoT)**: The model is prompted to "think step-by-step" before answering. (Improved logic, but no external interaction).
3. **ReAct**: The model thinks, acts (calls a tool), and then re-evaluates its plan based on the observation.

In [13]:
import os
from typing import List, Union
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_react_agent, AgentExecutor

load_dotenv()

# Central model configuration
model = "gemini-2.5-flash"
llm = ChatGoogleGenerativeAI(model=model, temperature=0)

print(f"ReAct environment initialized with model: {model}")

ReAct environment initialized with model: gemini-2.5-flash


## 2. Architecture: ReAct vs. Plan-and-Execute
Two primary architectures exist for complex task handling:

| Architecture | Approach | Strength | Weakness |
| :--- | :--- | :--- | :--- |
| **ReAct** | Iterative: Think -> Act -> Observe -> Repeat | Highly robust to unexpected tool outputs. | Can get stuck in loops; higher latency. |
| **Plan-and-Execute** | Sequential: Plan all steps -> Execute all -> Final Answer | Efficient for predictable tasks. | Brittle; fails if step 1 changes the requirements for step 5. |

In most agentic systems, **ReAct** is preferred because it allows the model to pivot when a tool returns an error or unexpected data.

## 3. The ReAct Anatomy
A standard ReAct prompt enforces a specific structural loop:

1. **Thought**: The model's internal reasoning about the current state.
2. **Action**: The specific tool to call.
3. **Action Input**: The parameters for that tool.
4. **Observation**: The data returned from the tool (provided by the system).
5. **Final Answer**: The ultimate response once reasoning is complete.

## 4. Manual ReAct Implementation (Demystification)
Before using high-level abstractions, understanding the pattern requires seeing the raw loop. This manual implementation shows how "Agency" is simply an iterative conversation where the system provides the "Observation" back to the model.

In [14]:
def custom_word_counter(text: str):
    return len(text.split())

prompt_state = """
Task: How many words are in the phrase 'Reasoning leads to action'?
Strategy: Use the word_counter tool and then provide the answer.

Thought: I need to count the words in the phrase.
Action: word_counter
Action Input: 'Reasoning leads to action'
"""

# Theoretical Observation Step (Simulating system feedback)
observation = custom_word_counter("Reasoning leads to action")

# Re-prompting with Observation
extended_prompt = f"{prompt_state}\nObservation: {observation}\nThought: I now have the count. The answer is 4.\nFinal Answer: 4"

print(extended_prompt)


Task: How many words are in the phrase 'Reasoning leads to action'?
Strategy: Use the word_counter tool and then provide the answer.

Thought: I need to count the words in the phrase.
Action: word_counter
Action Input: 'Reasoning leads to action'

Observation: 4
Thought: I now have the count. The answer is 4.
Final Answer: 4


## 5. Basic Automated Agents with LangChain
LangChain's `create_react_agent` automates this loop. It parses the model's "Action" strings, executes the corresponding Python function, and automatically appends the "Observation" for the next turn.

In [15]:
@tool
def calculate_string_length(text: str) -> int:
    """Returns the number of characters in a string. Input must be a raw string."""
    return len(text)

tools = [calculate_string_length]

template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: always start with a high-level reasoning plan
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

agent_executor.invoke({"input": "What is the length of 'Antigravity'?"})



> Entering new AgentExecutor chain...
always start with a high-level reasoning plan
Action: calculate_string_length
Action Input: Antigravity11I now know the final answer
Final Answer: 11

> Finished chain.


{'input': "What is the length of 'Antigravity'?", 'output': '11'}

## 6. Advanced: Self-Correction & Reasoning Guardrails
Complex reasoning requires the model to evaluate its own plan. By interleaving an internal 'Critique' phase within the Thought process, agents can self-correct logic errors before triggering a tool call.

In [16]:
reflect_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: Always begin by critiquing any potential logic errors in your current plan before formulating the next step.
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

reflect_prompt = PromptTemplate.from_template(reflect_template)
reflect_agent = create_react_agent(llm, tools, reflect_prompt)
reflect_executor = AgentExecutor(agent=reflect_agent, tools=tools, verbose=True, handle_parsing_errors=True)

reflect_executor.invoke({"input": "Calculate the length of the string 'Agentic AI'"})



> Entering new AgentExecutor chain...
Action: calculate_string_length
Action Input: Agentic AI10I now know the final answer
Final Answer: 10

> Finished chain.


{'input': "Calculate the length of the string 'Agentic AI'", 'output': '10'}

## 7. Advanced: Production Resilience
In production, agents encounter ambiguity and infinite loops. Using safety guardrails like `max_iterations` and 'Clarification' tools ensures that the agent fails gracefully rather than consuming infinite resources.

In [17]:
@tool
def get_user_clarification(missing_info: str) -> str:
    """Use this tool if the user's prompt is missing data or instructions. Explain what is needed."""
    return f"PAUSED: System needs more information about: {missing_info}"

prod_tools = [calculate_string_length, get_user_clarification]
prod_agent = create_react_agent(llm, prod_tools, prompt)
prod_executor = AgentExecutor(
    agent=prod_agent, 
    tools=prod_tools, 
    verbose=True, 
    handle_parsing_errors=True,
    max_iterations=3
)

print("Demonstrating Ambiguity Handling:")
prod_executor.invoke({"input": "Calculate the length of the secret word."})

Demonstrating Ambiguity Handling:


> Entering new AgentExecutor chain...
Action: get_user_clarification
Action Input: the secret wordPAUSED: System needs more information about: the secret wordAction: get_user_clarification
Action Input: the secret wordPAUSED: System needs more information about: the secret wordThought: The system is still paused, indicating that the secret word has not been provided by the user. I need the secret word to calculate its length. Therefore, I must continue to request this missing information.
Action: get_user_clarification
Action Input: the secret wordPAUSED: System needs more information about: the secret word

> Finished chain.


{'input': 'Calculate the length of the secret word.',
 'output': 'Agent stopped due to iteration limit or time limit.'}

## 8. Final Project: The Autonomous Decision Engine
This culmination integrates multi-hop reasoning, where the results of one logical step are used as inputs for the next. This requires the model to precisely orchestrate multiple tool calls to solve a multi-layered deduction.

In [18]:
@tool
def safe_calculator(expression: str) -> str:
    """Calculates math expressions. Input should be a string like '7 * 2' or '14 - 5'."""
    try:
        # Using a restricted scope for eval as a master-class safety demonstration
        result = eval(expression, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error in expression: {str(e)}"

@tool
def check_teenager(age: str) -> str:
    """Determines if an age (string or int) is between 13 and 19 inclusive."""
    try:
        val = int(float(age))
        return "is a Teenager" if 13 <= val <= 19 else "is not a Teenager"
    except:
        return "Error: Age must be a number"

final_tools = [safe_calculator, check_teenager]
final_agent = create_react_agent(llm, final_tools, prompt)
final_executor = AgentExecutor(agent=final_agent, tools=final_tools, verbose=True, handle_parsing_errors=True)

puzzle = "B is 7 years old. A is twice as old as B. Subtract 5 from A's age and determine if the result is a teenager."
print("\nSolving Multi-Hop Logic Puzzle:")
final_executor.invoke({"input": puzzle})


Solving Multi-Hop Logic Puzzle:


> Entering new AgentExecutor chain...
Thought: The question requires me to first calculate A's age, then subtract 5 from it, and finally check if the resulting age is a teenager.
First, calculate A's age: A is twice as old as B, and B is 7. So, A = 2 * 7.
Action: safe_calculator
Action Input: 2 * 714Now I have A's age, which is 14. The next step is to subtract 5 from A's age.
Action: safe_calculator
Action Input: 14 - 59Now I have A's age minus 5, which is 9. The final step is to determine if 9 is a teenager using the `check_teenager` tool.
Action: check_teenager
Action Input: 9is not a TeenagerI now know the final answer
Final Answer: 9 is not a Teenager

> Finished chain.


{'input': "B is 7 years old. A is twice as old as B. Subtract 5 from A's age and determine if the result is a teenager.",
 'output': '9 is not a Teenager'}

## Conclusion
The ReAct pattern transforms static generation into dynamic problem-solving. By interleaving reasoning and action, agents can interact with the real world, verify their own assumptions, and adjust to reality in real-time. This logic forms the core of modern autonomous agent architectures.